# ERP Factorial — 02: Explore

Collapses granular condition levels into meaningful groups and visualizes
condition overlays at candidate ROIs.

**Goal:** identify ROIs and time windows to commit in notebook 03.
**Requires:** Qt5 backend

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib
matplotlib.use('Qt5Agg')
import matplotlib.pyplot as plt
import mne
import numpy as np

from eeg_toolkit import load_config, find_subjects
from eeg_toolkit.evoked import get_evoked_path

# ── Update these paths ──
cfg     = load_config('../../../configs/your_experiment.yaml')
cfg_erp = load_config('../../../configs/your_erp_factorial.yaml')

subjects = find_subjects(cfg)
print(f"Analysis: {getattr(cfg_erp.erp, 'analysis_name', '(default)')}")

In [ ]:
# ── Define how to collapse granular conditions into groups ──
# Each group is a weighted average of the listed condition names.
# Condition names must match those defined in your factorial ERP config.
#
# Example: 2 (cue_type) × 2 (side) design, collapse to left/right per factor level
groupings = {
    'factor1_level1_left':  ['factor1_level1_pos1', 'factor1_level1_pos2'],
    'factor1_level1_right': ['factor1_level1_pos3', 'factor1_level1_pos4'],
    'factor1_level2_left':  ['factor1_level2_pos1', 'factor1_level2_pos2'],
    'factor1_level2_right': ['factor1_level2_pos3', 'factor1_level2_pos4'],
}

# ── Load and collapse per subject ──
window_name = 'your_window'   # must match a window name in cfg_erp
collapsed = {grp: [] for grp in groupings}

for subject in subjects:
    evo_path = get_evoked_path(cfg, subject, window_name, cfg_erp)
    if not evo_path.exists():
        continue
    subj = {e.comment: e for e in mne.read_evokeds(evo_path, verbose='WARNING')}

    for grp_name, components in groupings.items():
        evos = [subj[c] for c in components if c in subj]
        if len(evos) == len(components):
            combined = mne.combine_evoked(evos, weights='equal')
            combined.comment = grp_name
            collapsed[grp_name].append(combined)

print("Subjects per group:")
for grp, lst in collapsed.items():
    print(f"   {grp}: N={len(lst)}")

In [ ]:
# ── Visualize condition overlays at candidate ROIs ──
# ── Update ROIs and plot groups for your design ──
rois = {
    'left_roi':  ['Ch1', 'Ch2', 'Ch3'],
    'right_roi': ['Ch4', 'Ch5', 'Ch6'],
}

# Each entry: (title, [group_a, group_b]) to compare
plot_groups = [
    ('Factor1 Level1', ['factor1_level1_left', 'factor1_level1_right']),
    ('Factor1 Level2', ['factor1_level2_left', 'factor1_level2_right']),
]

for title, groups in plot_groups:
    evokeds_to_plot = {grp: collapsed[grp] for grp in groups}
    for roi_name, channels in rois.items():
        valid_ch = [c for c in channels if c in collapsed[groups[0]][0].ch_names]
        fig = mne.viz.plot_compare_evokeds(
            evokeds_to_plot,
            picks=valid_ch,
            combine='mean',
            title=f"{title} — {roi_name}",
            ci=0.95,
            show=False,
        )
        plt.show()